# Project: GooglePay Expense Sharing

- **Business Use Case:** Friends often pay for different things on a trip. I made this small project to find out who needs to pay whom at the end.
- **Description:** I used Alice, Bob and Carol as an example. They share accommodation and food expenses. The notebook calculates the final payments and shows a couple of simple charts.

## 1. Import libraries

I used Pandas to make a table of expenses, NumPy for one calculation, and Matplotlib for the charts.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

## 2. Create the expense-sharing class

I made one class to store the expenses and balance for each friend. A positive balance means that person should get money back. A negative balance means that person needs to pay.

In [ ]:
class ExpenseSharing:
    def __init__(self, friends):
        self.friends = friends
        self.balances = {friend: 0 for friend in friends}
        self.expenses = []

    def add_expense(self, payer, amount, participants, description, category):
        split_amount = amount / len(participants)

        # Save the expense so it can be analysed later using Pandas.
        self.expenses.append({
            'Description': description,
            'Payer': payer,
            'Amount': amount,
            'Category': category
        })

        # The payer paid the complete amount.
        self.balances[payer] += amount

        # Each participant pays only their equal share.
        for person in participants:
            self.balances[person] -= split_amount

    def show_balances(self):
        for friend, balance in self.balances.items():
            if balance > 0:
                print(f'{friend} should receive Rs. {balance:.2f}')
            elif balance < 0:
                print(f'{friend} owes Rs. {-balance:.2f}')
            else:
                print(f'{friend} is settled up')

    def calculate_settlement(self):
        creditors = []
        debtors = []

        for friend, balance in self.balances.items():
            if balance > 0:
                creditors.append([friend, round(balance, 2)])
            elif balance < 0:
                debtors.append([friend, round(-balance, 2)])

        payments = []
        while debtors and creditors:
            debtor, debt_amount = debtors.pop()
            creditor, credit_amount = creditors.pop()

            payment = min(debt_amount, credit_amount)
            payments.append((debtor, creditor, payment))

            if debt_amount > payment:
                debtors.append([debtor, round(debt_amount - payment, 2)])
            if credit_amount > payment:
                creditors.append([creditor, round(credit_amount - payment, 2)])

        return payments

## 3. Add the trip expenses

For this example, all three friends share every bill equally. The total comes to Rs. 15,000, so each person should finally pay Rs. 5,000.

In [ ]:
friends = ['Alice', 'Bob', 'Carol']
expense_sharing = ExpenseSharing(friends)

expense_sharing.add_expense('Alice', 9000, friends, 'Accommodation', 'Stay')
expense_sharing.add_expense('Bob', 1500, friends, 'Day 1 dinner', 'Food')
expense_sharing.add_expense('Carol', 1200, friends, 'Breakfast and lunch', 'Food')
expense_sharing.add_expense('Bob', 900, friends, 'Day 2 lunch', 'Food')
expense_sharing.add_expense('Alice', 1800, friends, 'Day 2 dinner', 'Food')
expense_sharing.add_expense('Carol', 600, friends, 'Day 3 breakfast', 'Food')

## 4. Check balances and final settlement

In [ ]:
expense_sharing.show_balances()

print('\nFinal Settlement:')
for debtor, creditor, amount in expense_sharing.calculate_settlement():
    print(f'{debtor} pays {creditor}: Rs. {amount:.2f}')

## 5. Create a data table

For the data-preprocessing part, I changed the Python list of expenses into a Pandas DataFrame. This makes the data easier to look at and group.

In [ ]:
expense_data = pd.DataFrame(expense_sharing.expenses)
expense_data

## 6. Simple data analysis

Here I used `groupby()` to add the amounts by category and by the person who paid. I also found the average expense using NumPy.

In [ ]:
spending_by_category = expense_data.groupby('Category')['Amount'].sum()
spending_by_payer = expense_data.groupby('Payer')['Amount'].sum()
average_expense = np.mean(expense_data['Amount'])

print('Spending by category:')
print(spending_by_category)

print('\nAmount paid by each friend:')
print(spending_by_payer)

print(f'\nAverage expense: Rs. {average_expense:.2f}')

## 7. Visualize spending patterns

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

spending_by_category.plot(kind='bar', ax=axes[0], color='skyblue')
axes[0].set_title('Spending by Category')
axes[0].set_xlabel('Category')
axes[0].set_ylabel('Amount (Rs.)')
axes[0].tick_params(axis='x', rotation=0)

spending_by_payer.plot(kind='bar', ax=axes[1], color='lightgreen')
axes[1].set_title('Amount Paid by Each Friend')
axes[1].set_xlabel('Friend')
axes[1].set_ylabel('Amount (Rs.)')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## 8. Results and insights

- The group spent **Rs. 15,000**, so each friend should pay **Rs. 5,000**.
- Alice paid Rs. 10,800, so she needs to get Rs. 5,800 back.
- Bob pays Alice Rs. 2,600 and Carol pays Alice Rs. 3,200.
- From the chart I can see that accommodation is the biggest expense, and Alice paid the most.
- If somebody joins an expense but does not pay at that time, they are still included in the participants list. Their balance shows what they owe.
- If I wanted to improve the project, I could add refunds as negative expenses and allow users to enter their own expenses.

## 9. Conclusion

This project helped me understand how Python can be used with simple data analysis. It calculates fair shares, shows the final payments, and makes graphs from the same expense data.